In [0]:
# Gold alerts configuration
spark.sql("USE CATALOG dbw_fleet_telemetry_dev")
storage_account = "stfleettelemetryalvin"
storage_key     = "YOUR_STORAGE_ACCOUNT_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

CHECKPOINT_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_checkpoints/fleet_gold"

# Verify Silver has data
spark.sql("SELECT COUNT(*) as silver_rows FROM silver.vehicle_window_stats").show()

+-----------+
|silver_rows|
+-----------+
|        290|
+-----------+



In [0]:
# Create Gold alerts table
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.fleet_alerts (
        alert_id            STRING,
        vehicle_id          STRING,
        driver_name         STRING,
        route_name          STRING,
        home_depot          STRING,
        alert_type          STRING,
        severity            STRING,
        alert_status        STRING,
        avg_speed_kmh       DOUBLE,
        rated_max_speed     LONG,
        speed_pct_of_max    DOUBLE,
        avg_engine_temp_c   DOUBLE,
        min_fuel_pct        FLOAT,
        anomaly_event_count LONG,
        last_lat            DOUBLE,
        last_lon            DOUBLE,
        window_start        TIMESTAMP,
        window_end          TIMESTAMP,
        alert_raised_at     TIMESTAMP,
        last_updated_at     TIMESTAMP,
        resolved_at         TIMESTAMP
    )
    USING DELTA
""")

print("Gold alerts table created")
spark.sql("DESCRIBE gold.fleet_alerts").show(truncate=False)

Gold alerts table created
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|alert_id           |string   |NULL   |
|vehicle_id         |string   |NULL   |
|driver_name        |string   |NULL   |
|route_name         |string   |NULL   |
|home_depot         |string   |NULL   |
|alert_type         |string   |NULL   |
|severity           |string   |NULL   |
|alert_status       |string   |NULL   |
|avg_speed_kmh      |double   |NULL   |
|rated_max_speed    |bigint   |NULL   |
|speed_pct_of_max   |double   |NULL   |
|avg_engine_temp_c  |double   |NULL   |
|min_fuel_pct       |float    |NULL   |
|anomaly_event_count|bigint   |NULL   |
|last_lat           |double   |NULL   |
|last_lon           |double   |NULL   |
|window_start       |timestamp|NULL   |
|window_end         |timestamp|NULL   |
|alert_raised_at    |timestamp|NULL   |
|last_updated_at    |timestamp|NULL   |
+-------------------+---------+-------+
only showing t

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Alert thresholds
SPEED_BREACH_MULTIPLIER = 1.15
ENGINE_TEMP_WARNING     = 95.0
ENGINE_TEMP_CRITICAL    = 110.0
FUEL_LOW_PCT            = 15.0
ANOMALY_THRESHOLD       = 3

print("Thresholds defined")

Thresholds defined


In [0]:
def merge_alerts_to_gold(batch_df, batch_id):
    if batch_df.isEmpty():
        print(f"[Batch {batch_id}] Empty - skipping")
        return

    print(f"[Batch {batch_id}] Processing {batch_df.count()} rows...")

    alerts = (
        batch_df
        .withColumn("alert_status",
            F.when(F.col("severity") != "NORMAL", "OPEN").otherwise("RESOLVED")
        )
        .withColumn("alert_id", F.expr("uuid()"))
        .withColumn("alert_raised_at",
            F.when(F.col("alert_status") == "OPEN", F.current_timestamp())
        )
        .withColumn("last_updated_at", F.current_timestamp())
        .withColumn("resolved_at",
            F.when(F.col("alert_status") == "RESOLVED", F.current_timestamp())
        )
    )

    window_spec = Window.partitionBy("vehicle_id").orderBy(F.col("window_end").desc())
    deduped = (
        alerts
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    open_count     = deduped.filter(F.col("alert_status") == "OPEN").count()
    resolved_count = deduped.filter(F.col("alert_status") == "RESOLVED").count()
    print(f"[Batch {batch_id}] OPEN={open_count} | RESOLVED={resolved_count}")

    gold = DeltaTable.forName(spark, "gold.fleet_alerts")
    (
        gold.alias("g")
        .merge(deduped.alias("n"), "g.vehicle_id = n.vehicle_id")
        .whenMatchedUpdate(set={
            "alert_type":          "n.alert_type",
            "severity":            "n.severity",
            "alert_status":        "n.alert_status",
            "avg_speed_kmh":       "n.avg_speed_kmh",
            "rated_max_speed":     "n.rated_max_speed",
            "speed_pct_of_max":    "n.speed_pct_of_max",
            "avg_engine_temp_c":   "n.avg_engine_temp_c",
            "min_fuel_pct":        "n.min_fuel_pct",
            "anomaly_event_count": "n.anomaly_event_count",
            "last_lat":            "n.last_lat",
            "last_lon":            "n.last_lon",
            "window_start":        "n.window_start",
            "window_end":          "n.window_end",
            "last_updated_at":     "n.last_updated_at",
            "resolved_at":         "n.resolved_at",
            "alert_raised_at":     "n.alert_raised_at",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"[Batch {batch_id}] MERGE complete")

print("Function defined")

Function defined


In [0]:
silver_stream = (
    spark.readStream
    .format("delta")
    .table("silver.vehicle_window_stats")
)

gold_query = (
    silver_stream
    .writeStream
    .foreachBatch(merge_alerts_to_gold)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="30 seconds")
    .start()
)

print(f"Gold stream started")
print(f"Query ID: {gold_query.id}")

Gold stream started
Query ID: 2d9ec833-eb7e-4eef-b5e3-1efccd81dc10


In [0]:
spark.sql("""
    SELECT 
        vehicle_id,
        driver_name,
        route_name,
        alert_status,
        severity,
        alert_type,
        ROUND(avg_speed_kmh, 1)    AS avg_speed,
        ROUND(avg_engine_temp_c, 1) AS engine_temp,
        ROUND(min_fuel_pct, 1)     AS fuel_pct,
        last_updated_at
    FROM gold.fleet_alerts
    ORDER BY
        CASE severity 
            WHEN 'CRITICAL' THEN 1 
            WHEN 'HIGH' THEN 2 
            WHEN 'MEDIUM' THEN 3 
            WHEN 'LOW' THEN 4 
            ELSE 5 
        END
""").show(truncate=False)

+----------+---------------+--------------------------+------------+--------+---------------+---------+-----------+--------+-----------------------+
|vehicle_id|driver_name    |route_name                |alert_status|severity|alert_type     |avg_speed|engine_temp|fuel_pct|last_updated_at        |
+----------+---------------+--------------------------+------------+--------+---------------+---------+-----------+--------+-----------------------+
|VH002     |Robert Carter  |Bangalore-Chennai Corridor|OPEN        |CRITICAL|engine_overheat|99.8     |96.4       |1.1     |2026-07-02 19:42:34.784|
|VH004     |Thomas Harris  |Bangalore-Hyderabad NH44  |OPEN        |MEDIUM  |speed_breach   |72.6     |96.6       |1.0     |2026-07-02 19:42:34.784|
|VH007     |Matthew Wilson |Bangalore-Hyderabad NH44  |OPEN        |MEDIUM  |speed_breach   |74.4     |96.7       |1.9     |2026-07-02 19:42:34.784|
|VH001     |James Mitchell |Bangalore-Mysore Express  |RESOLVED    |NORMAL  |normal         |79.8     |96.

In [0]:
spark.sql("""
    SELECT vehicle_id, driver_name, route_name,
           alert_status, severity, alert_type,
           ROUND(avg_speed_kmh,1) as avg_speed,
           ROUND(min_fuel_pct,1) as fuel_pct
    FROM gold.fleet_alerts
    ORDER BY CASE severity 
        WHEN 'CRITICAL' THEN 1 
        WHEN 'MEDIUM' THEN 2 
        ELSE 3 END
""").show(truncate=False)

+----------+---------------+--------------------------+------------+--------+---------------+---------+--------+
|vehicle_id|driver_name    |route_name                |alert_status|severity|alert_type     |avg_speed|fuel_pct|
+----------+---------------+--------------------------+------------+--------+---------------+---------+--------+
|VH002     |Robert Carter  |Bangalore-Chennai Corridor|OPEN        |CRITICAL|engine_overheat|99.8     |1.1     |
|VH004     |Thomas Harris  |Bangalore-Hyderabad NH44  |OPEN        |MEDIUM  |speed_breach   |72.6     |1.0     |
|VH007     |Matthew Wilson |Bangalore-Hyderabad NH44  |OPEN        |MEDIUM  |speed_breach   |74.4     |1.9     |
|VH001     |James Mitchell |Bangalore-Mysore Express  |RESOLVED    |NORMAL  |normal         |79.8     |2.1     |
|VH003     |William Turner |Bangalore-Mysore Express  |RESOLVED    |NORMAL  |normal         |80.1     |1.3     |
|VH005     |Daniel Evans   |Bangalore-Pune Highway    |RESOLVED    |NORMAL  |normal         |98.

In [0]:
spark.sql("""
    SELECT alert_type, severity, COUNT(*) as count
    FROM bronze.raw_telemetry
    GROUP BY alert_type, severity
    ORDER BY count DESC
""").show()

+---------------+--------+------+
|     alert_type|severity| count|
+---------------+--------+------+
|         normal|  NORMAL|114229|
|   speed_breach|  MEDIUM| 19635|
|engine_overheat|CRITICAL| 14566|
|  fuel_critical|    HIGH|  9741|
|route_deviation|  MEDIUM|  4859|
+---------------+--------+------+



In [0]:
spark.sql("""
    SELECT vehicle_id, driver_name, window_start, window_end,
           ROUND(avg_speed_kmh,1) as avg_speed,
           alert_type, severity, event_count
    FROM silver.vehicle_window_stats
    ORDER BY window_start DESC
    LIMIT 10
""").show(truncate=False)

+----------+---------------+-------------------+-------------------+---------+---------------+--------+-----------+
|vehicle_id|driver_name    |window_start       |window_end         |avg_speed|alert_type     |severity|event_count|
+----------+---------------+-------------------+-------------------+---------+---------------+--------+-----------+
|VH002     |Robert Carter  |2026-07-02 19:39:00|2026-07-02 19:40:00|99.8     |engine_overheat|CRITICAL|410        |
|VH010     |Richard Moore  |2026-07-02 19:39:00|2026-07-02 19:40:00|100.1    |normal         |NORMAL  |410        |
|VH007     |Matthew Wilson |2026-07-02 19:39:00|2026-07-02 19:40:00|74.4     |speed_breach   |MEDIUM  |410        |
|VH009     |David Anderson |2026-07-02 19:39:00|2026-07-02 19:40:00|109.9    |normal         |NORMAL  |410        |
|VH003     |William Turner |2026-07-02 19:39:00|2026-07-02 19:40:00|80.1     |normal         |NORMAL  |410        |
|VH005     |Daniel Evans   |2026-07-02 19:39:00|2026-07-02 19:40:00|98.4